# GTEx model with different priors using CLAMP

💡 **Environment:** `clamp-analyses`  

This notebook create different CLAMP GTEx models using different biological pathway priors. It loads a pre-computed CLAMP base model, SVD decomposition, and preprocessed GTEx data, then systematically evaluates distinct biological knowledge sources as priors. Each prior is run independently through CLAMPfull to generate and a final combined model incorporates all priors simultaneously.

## Load libraries

In [1]:
library(bigstatsr)
library(data.table)
library(dplyr)
library(rsvd)
library(glmnet)
library(Matrix)
library(knitr)
library(here)
library(PLIER)
library(CLAMP)

source(here("config.R"))

set.seed(config$GTEx$RANDOM_SVD_SEED)


Attaching package: ‘dplyr’


The following objects are masked from ‘package:data.table’:

    between, first, last


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Loading required package: Matrix

Loaded glmnet 4.1-10

here() starts at /home/msubirana/Documents/pivlab/clamp-analyses

Loading required package: RColorBrewer

Loading required package: gplots


---------------------
gplots 3.3.0 loaded:
  * Use citation('gplots') for citation info.
  * Homepage: https://talgalili.github.io/gplots/
  * Report issues: https://github.com/talgalili/gplots/issues
  * Ask questions: https://stackoverflow.com/questions/tagged/gplots
  * Suppress this message with: suppressPackageStartupMessages(library(gplots))
---------------------



Attaching package: ‘gplots’


The following object is masked from ‘package:stats’:

    lowess


Loading required package: pheatmap

Loadin

## Output directory

In [2]:
output_data_dir <- config$GTEx$OUTPUT_DIR
dir.create(output_data_dir, showWarnings = FALSE, recursive = TRUE)

# Load GTEx CLAMP base model

In [3]:
gtex_baseRes <- readRDS(file.path(output_data_dir, "CLAMPbase.rds"))

In [4]:
gtex_svdRes <- readRDS(file.path(output_data_dir, "gtex_svdRes.rds"))

In [5]:
CLAMP_K_gtex <- readRDS(file.path(output_data_dir, "CLAMP_K_gtex.rds"))

In [6]:
gtex_fbm_filt <- readRDS(file.path(output_data_dir, "gtex_fbm_filt.rds"))

In [7]:
gtex_genes <- readRDS(file.path(output_data_dir, "gtex_genes.rds"))

In [8]:
samples <- readRDS(file.path(output_data_dir, "gtex_samples.rds"))

In [9]:
# Vector of prior URLs
prior_urls <- c(
  "https://maayanlab.cloud/Enrichr/geneSetLibrary?mode=text&libraryName=KEGG_2021_Human",
  "https://maayanlab.cloud/Enrichr/geneSetLibrary?mode=text&libraryName=GO_Biological_Process_2025",
  "https://maayanlab.cloud/Enrichr/geneSetLibrary?mode=text&libraryName=CellMarker_2024"
)

# Names are exactly what's after libraryName=
names(prior_urls) <- sub(".*libraryName=", "", prior_urls)

results <- list()

for (prior_name in names(prior_urls)) {
  url <- prior_urls[[prior_name]]
  message(">>> Running CLAMP for prior: ", prior_name)

  try({
    gmt <- getGMT(url)
    gmt_list <- list()
    gmt_list[[prior_name]] <- gmt

    pathMat   <- gmtListToSparseMat(gmt_list)
    matched   <- getMatchedPathwayMat(pathMat, gtex_genes)
    if (is.null(matched) || ncol(matched) == 0) {
      warning("No matched pathways for ", prior_name, "; skipping.")
      next
    }

    fullRes <- CLAMPfull(
      Y                 = gtex_fbm_filt,
      priorMat          = as.matrix(matched),
      svdres            = gtex_svdRes,
      clamp.base.result = gtex_baseRes,
      clamp_k           = CLAMP_K_gtex,
      doCrossval        = TRUE,
      trace             = TRUE,
      use_cpp           = TRUE
    )

    if (!is.null(fullRes$B)) colnames(fullRes$B) <- samples
    if (!is.null(fullRes$Z)) colnames(fullRes$Z) <- paste0("LV", seq_len(ncol(fullRes$Z)))
    if (!is.null(fullRes$summary)) {
      fullRes$summary <- fullRes$summary |>
        dplyr::rename(LV = LV_index) |>
        dplyr::mutate(LV = paste0("LV", LV))
    }

    out_file <- file.path(output_data_dir, sprintf("gtex_%s_CLAMP.rds", prior_name))
    saveRDS(fullRes, file = out_file)
    results[[prior_name]] <- fullRes

    message("Saved: ", out_file)
  }, silent = FALSE)
}

invisible(results)

>>> Running CLAMP for prior: KEGG_2021_Human

Auto-detected name: KEGG_2021_Human

Using cached file for KEGG_2021_Human



There are 6409 genes in the intersection between data and prior

Removing 10 pathways

** CLAMPfull **

using provided CLAMPbase result

CLAMP k is set to 412

L1=45.2778145717356; L2=135.833443715207



## CLAMPfull C2CP (local file)

In [9]:
data_path <- here::here('data/archs4')

c2_gmt <- CLAMP:::read_gmt(file.path(data_path, "c2.cp.v2026.1.Hs.symbols.gmt"))
names(c2_gmt) <- paste0("C2CP_", names(c2_gmt))

c2_pathMat <- gmtListToSparseMat(list(C2CP = c2_gmt))
c2_matched  <- getMatchedPathwayMat(c2_pathMat, gtex_genes)

c2_fullRes <- CLAMPfull(
  Y                 = gtex_fbm_filt,
  priorMat          = as.matrix(c2_matched),
  svdres            = gtex_svdRes,
  clamp.base.result = gtex_baseRes,
  clamp_k           = CLAMP_K_gtex,
  doCrossval        = TRUE,
  trace             = TRUE,
  use_cpp           = TRUE
)

if (!is.null(c2_fullRes$B)) colnames(c2_fullRes$B) <- samples
if (!is.null(c2_fullRes$Z)) colnames(c2_fullRes$Z) <- paste0("LV", seq_len(ncol(c2_fullRes$Z)))
if (!is.null(c2_fullRes$summary)) {
  c2_fullRes$summary <- c2_fullRes$summary |>
    dplyr::rename(LV = LV_index) |>
    dplyr::mutate(LV = paste0("LV", LV))
}

saveRDS(c2_fullRes, file = file.path(output_data_dir, "gtex_C2CP_CLAMP.rds"))
message("Saved: gtex_C2CP_CLAMP.rds")

There are 11250 genes in the intersection between data and prior

Removing 980 pathways

** CLAMPfull **

using provided CLAMPbase result

CLAMP k is set to 412

L1=45.2778145717356; L2=135.833443715207

Progress 1 / 30 | Bdiff=0.000454

Progress 2 / 30 | Bdiff=0.037459

Progress 3 / 30 | Bdiff=0.015090

Estimated total runtime: ~8.6 min

Progress 4 / 30 | Bdiff=0.012629

Progress 5 / 30 | Bdiff=0.008692

Progress 6 / 30 | Bdiff=0.006472

Progress 7 / 30 | Bdiff=0.005402

Progress 8 / 30 | Bdiff=0.004966

Progress 9 / 30 | Bdiff=0.004979

Progress 10 / 30 | Bdiff=0.004901

Progress 11 / 30 | Bdiff=0.004951

Progress 12 / 30 | Bdiff=0.004580

Progress 13 / 30 | Bdiff=0.003981

Progress 14 / 30 | Bdiff=0.004044

Progress 15 / 30 | Bdiff=0.003861

Progress 16 / 30 | Bdiff=0.003706

Progress 17 / 30 | Bdiff=0.003725

Progress 18 / 30 | Bdiff=0.003420

Progress 19 / 30 | Bdiff=0.003297

Progress 20 / 30 | Bdiff=0.003160

Progress 21 / 30 | Bdiff=0.002968

Progress 22 / 30 | Bdiff=0.003099



In [10]:
# priors
gtex_gmtList <- list(
  KEGG = getGMT("https://maayanlab.cloud/Enrichr/geneSetLibrary?mode=text&libraryName=KEGG_2021_Human"),
  BP = getGMT("https://maayanlab.cloud/Enrichr/geneSetLibrary?mode=text&libraryName=GO_Biological_Process_2025"),
  CellMarker = getGMT("https://maayanlab.cloud/Enrichr/geneSetLibrary?mode=text&libraryName=CellMarker_2024")

)

# prefix each gene‐set name with its library to guarantee uniqueness
for(lib in names(gtex_gmtList)) {
  names(gtex_gmtList[[lib]]) <- paste0(lib, "_", names(gtex_gmtList[[lib]]))
}

gtex_pathMat <- gmtListToSparseMat(gtex_gmtList)
gtex_matched <- getMatchedPathwayMat(gtex_pathMat, gtex_genes)

# CLAMPfull
gtex_fullRes <- CLAMPfull(
  Y                 = gtex_fbm_filt,
  priorMat          = as.matrix(gtex_matched),
  svdres            = gtex_svdRes,
  clamp.base.result = gtex_baseRes,
  clamp_k           = CLAMP_K_gtex,
  doCrossval        = TRUE,
  trace             = TRUE,
  use_cpp           = TRUE
)

# Fix colnames and rownames
colnames(gtex_fullRes$B) <- samples
colnames(gtex_fullRes$Z) <- paste0('LV', seq_len(ncol(gtex_fullRes$Z)))
gtex_fullRes$summary <- gtex_fullRes$summary %>%
    dplyr::rename(LV = LV_index)  %>% 
    dplyr::mutate(LV = paste0('LV', LV))

# save
saveRDS(gtex_fullRes, file = file.path(output_data_dir, "gtex_all_CLAMP.rds"))

Auto-detected name: KEGG_2021_Human

Using cached file for KEGG_2021_Human



Auto-detected name: GO_Biological_Process_2025

Using cached file for GO_Biological_Process_2025

Auto-detected name: CellMarker_2024

Using cached file for CellMarker_2024

There are 14239 genes in the intersection between data and prior

Removing 2995 pathways

** CLAMPfull **

using provided CLAMPbase result

CLAMP k is set to 412

L1=45.2778145717356; L2=135.833443715207

Progress 1 / 30 | Bdiff=0.000454

Progress 2 / 30 | Bdiff=0.037502

Progress 3 / 30 | Bdiff=0.015047

Estimated total runtime: ~8.0 min

Progress 4 / 30 | Bdiff=0.012777

Progress 5 / 30 | Bdiff=0.008986

Progress 6 / 30 | Bdiff=0.006596

Progress 7 / 30 | Bdiff=0.005377

Progress 8 / 30 | Bdiff=0.005152

Progress 9 / 30 | Bdiff=0.005127

Progress 10 / 30 | Bdiff=0.004858

Progress 11 / 30 | Bdiff=0.004953

Progress 12 / 30 | Bdiff=0.004512

Progress 13 / 30 | Bdiff=0.004030

Progress 14 / 30 | Bdiff=0.003985

Progress 15 / 30 | Bdiff=0.003742

Progress 16 / 30 | Bdiff=0.003593

Progress 17 / 30 | Bdiff=0.003729

